In [ ]:
!pip install seaborn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import re
import pandas as pd
import seaborn as sns
import numpy as np


# Helper Functions

In [ ]:
import logging

# Configure le logger racine pour afficher les logs dans le notebook
logging.basicConfig(
    level=logging.INFO,  # ou DEBUG selon le niveau souhaité
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Exemple d'utilisation
logging.info("Logger activé dans le notebook Jupyter")

In [ ]:
def plot_by_source(df, source, title, column_name):
    # Filtrer par source
    df_source = df[df['source'] == source]
    
    # Calcul en pourcentage
    contract_pct = df_source[column_name].value_counts(normalize=True).mul(100).round(1)
    
    # Graphique
    fig, ax = plt.subplots(figsize=(10, 5))
    contract_pct.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    
    # Afficher les % sur les barres
    for i, v in enumerate(contract_pct):
        ax.text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

    ax.set_title(f"{title} — {source}", fontsize=13, fontweight='bold')
    #ax.set_title(f"Répartition des types de contrat — {source}", fontsize=13, fontweight='bold')
    ax.set_xlabel(column_name)
    ax.set_ylabel("Pourcentage (%)")
    ax.set_ylim(0, contract_pct.max() + 10)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


# Accessing Storage Files: Bronze → Silver
Explore how to access and manipulate job files stored in the bronze and silver layers using `get_storage_from_env()` and the project environment configuration.

In [ ]:
# Map du python path dans le docker
#import sys, os
#sys.path.insert(0, os.path.abspath('../..'))  # remonte à la racine du projet
#from src.config.env import load_project_env

## Load Project Environment
Import and load the project environment configuration to initialize environment variables.

In [ ]:
# Deactivate warning
import warnings
warnings.filterwarnings('ignore')

# Load project environment
from src.config.env import load_project_env
load_project_env()  # Safe to call multiple times (idempotent)
print("✅ Project environment loaded successfully")

from src.config.env import load_project_env
load_project_env()  # safe à rappeler (idempotent)

## Create storage connections for silver layers using `get_storage_from_env()` for the 'welcometothejungle' source.

In [ ]:
from src.storage.storage import get_storage_from_env
import src.utils.merge_dataset_utils as merge_utils
#import logging
#logger = logging.getLogger(__name__)

storage_wttj = get_storage_from_env("silver", "merged")

## Load wttj parquet file with helper

In [ ]:
# All
#df = merge_utils.read_wttj_parquet_file_to_df(storage_wttj,"")
df = merge_utils.read_wttj_parquet_file_to_df(storage_wttj,"merged_dt=2026-03-05_ft_dt=2026-03-03_wttj_dt=2026-03-05")


In [ ]:
df.info()

# Ficher Silver Merge déjà normalisé

In [ ]:
# Vérification de contract_normalized
plot_by_source(df, 'FT', 'Répartition des types de contrat' ,  'contract_normalized')
plot_by_source(df, 'WTTJ','Répartition des types de contrat', 'contract_normalized')

plot_by_source(df, 'FT', 'Répartition de l\'expérience',  'experience_normalized')
plot_by_source(df, 'WTTJ', 'Répartition de l\'expérience' , 'experience_normalized')



In [ ]:
# ==========
# Contrat
# ==========
display(df.groupby('source')['contract_type'].value_counts().head(10))

# ==========
# Experience
# ==========
# Libellé
display(df.groupby('source')['experience_normalized'].value_counts().head(10))
# Index associé
display(df.groupby('source')['experience_index'].value_counts().head(10))

# Original mixed values  
#display(df.groupby('source')['experience_detail'].value_counts().head(10))
#display(df.groupby('source')['experience_source_composite'].value_counts().head(10))

# ==========
# Salaires
# ==========
display(df.groupby('source')['salary_min'].value_counts().head(10))
display(df.groupby('source')['salary_max'].value_counts().head(10))

display(df.groupby('source')['yearly_min'].value_counts().head(10))
display(df.groupby('source')['yearly_max'].value_counts().head(10))





In [ ]:
#df_copy = df.copy()
df = merge_utils.normalize_salary(df)

In [ ]:
df[df['salary_min_computed'] < 100][['salary_min_computed', 'yearly_min', 'periodicity']]

In [ ]:
def plot_salary(df, column_name):

    salaries = df_copy[column_name].dropna()
    salaries = salaries[salaries > 0]
   
    # Histogramme complet
    plt.figure(figsize=(14, 4))
    sns.histplot(df_copy[column_name].dropna(), bins=200, kde=True)
    plt.title(f'Distribution complète des salaires {column_name} (avec outliers)')
    plt.xlabel('Salaire minimum')
    ax = plt.gca()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}".replace(',', ' ')))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}".replace(',', ' ')))
    plt.show()

    # Scatterplot pour visualiser la dispersion et les outliers
    plt.figure(figsize=(14, 4))
    plt.scatter(range(len(df_copy)), df_copy[column_name].sort_values())
    plt.title(f'Scatterplot des salaires {column_name} (outliers visibles à droite)')
    plt.xlabel('Index')
    plt.ylabel('Salaire minimum')
    ax = plt.gca()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}".replace(',', ' ')))
    plt.show()
     
    plt.figure(figsize=(12, 4))
    sns.scatterplot(x=np.arange(len(salaries)), y=salaries, alpha=0.5, label='Salaires')
    plt.axhline(salaries.mean(), color='red', linestyle='--', label='Moyenne')
    plt.axhline(salaries.median(), color='green', linestyle='-.', label='Médiane')
    plt.legend()
    plt.title(f'Nuage de points des salaires avec moyenne et médiane ({column_name})')
    plt.xlabel('Index')
    plt.ylabel('Salaire')
    ax = plt.gca()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}".replace(',', ' ')))
    plt.show()
        
    # Calcul des déciles
    deciles = np.percentile(salaries, np.arange(0, 110, 10))
    # Supprimer les doublons de déciles
    unique_deciles = np.unique(deciles)
    labels = [f"{int(unique_deciles[i])}-{int(unique_deciles[i+1])}" for i in range(len(unique_deciles)-1)]
    df_decile = pd.cut(salaries, bins=unique_deciles, labels=labels, include_lowest=True)
    decile_counts = df_decile.value_counts().sort_index()
    
    # Box Plot déciles
    plt.figure(figsize=(10, 5))
    sns.barplot(x=decile_counts.index, y=decile_counts.values)
    plt.title(f"Répartition des salaires par décile ({column_name})")
    plt.xlabel("Décile de salaire (€)")
    plt.ylabel("Nombre d'offres")
    plt.xticks(rotation=45)
    ax = plt.gca()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}".replace(',', ' ')))
    plt.show()
    
    # Violin
    plt.figure(figsize=(6, 4))
    sns.violinplot(y=salaries)
    plt.title(f'Violin plot des salaires ({column_name})')
    plt.ylabel('Salaire')
    ax = plt.gca()    
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}".replace(',', ' ')))
    plt.show()

    # Box Plot quartile 
    plt.figure(figsize=(6, 4))
    ax = sns.boxplot(y=salaries)
    quartiles = np.percentile(salaries, [25, 50, 75])
    for q, label in zip(quartiles, ['Q1 (25%)', 'Médiane (50%)', 'Q3 (75%)']):
        ax.axhline(q, color='red', linestyle='--', alpha=0.7)
        ax.text(0, q, f'{label}: {int(q)}', color='red', va='bottom', ha='left')
    plt.title(f'Boxplot des salaires avec quartiles ({column_name})')
    plt.ylabel('Salaire')
    ax = plt.gca()    
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}".replace(',', ' ')))
    plt.show()
    

In [ ]:
#display(df[df['yearly_min'].isna()].head(1))
#display(df[ df['yearly_max'].isna()].head(1))

#display(df[ df['salary_max']>150000][['source','title','description','salary_min','salary_max','yearly_min','yearly_max','salary_periodicity','periodicity']].head(5))
# Analyse salary min 
salary_min_counts = df.groupby('salary_min').size().reset_index(name='count')
display(salary_min_counts.sort_values('count', ascending=False).head(10))

# Analyse salary max 
salary_max_counts = df.groupby('salary_max').size().reset_index(name='count')
display(salary_max_counts.sort_values('count', ascending=False).head(50))


In [ ]:
# Analyse salary min 
salary_min_counts = df.groupby('salary_min_computed').size().reset_index(name='count')
display(salary_min_counts.sort_values('count', ascending=False).head(10))

# Analyse salary max 
salary_max_counts = df.groupby('salary_max_computed').size().reset_index(name='count')
display(salary_max_counts.sort_values('count', ascending=False).head(50))


In [ ]:
plot_salary(df, "salary_min")


In [ ]:
plot_salary(df, "salary_min_computed")

In [ ]:
plot_salary(df, "salary_max")

In [ ]:
plot_salary(df, "salary_max_computed")

In [ ]:
if df is not None and not df.empty:
    merge_utils.print_statistics_salaries(df)
else    :
    logger.warning("⚠️ Aucune donnée chargée pour les statistiques FT/WTTJ fusionnées")

## Contracts

In [ ]:
# contract_counts = df['contract_type'].value_counts()
contract_counts_by_source = df.groupby('source')['contract_type'].value_counts()

### Avant normalisation

In [ ]:
display(contract_counts_by_source.head(30))
#print(f" Libellé unique = {len(contract_counts_by_source)}")


### Après normalisation 

In [ ]:
def normalize_contracts(df, patterns):
    """
    Normalise les types de contrat :
    - Extrait le type principal (CDI, CDD, Intérim, etc.)
    - Extrait le détail (durée, précision)
    - Stocke dans contract_normalized et contract_detail
    """
    def extract_type(value):
        if pd.isna(value):
            return 'Inconnu'
        for pattern, label in patterns:
            if re.search(pattern, str(value)):
                return label
        return str(value)  # garder la valeur originale si pas de match

    def extract_detail(value):
        if pd.isna(value):
            return None
        # Supprimer le pattern trouvé et retourner le reste
        remaining = str(value)
        for pattern, label in patterns:
            remaining = re.sub(pattern, '', remaining).strip()
        # Nettoyer les séparateurs résiduels (-, :, espaces)
        remaining = re.sub(r'^[\s\-:]+|[\s\-:]+$', '', remaining)
        return remaining if remaining else None
    
    df = df.copy()
    df['contract_normalized'] = df['contract_type'].apply(extract_type)
    df['contract_detail']     = df['contract_type'].apply(extract_detail)
    
    return df


In [ ]:
# Mapping des patterns => type normalisé
patterns = [
    (r'(?i)cdi',                'CDI'),
    (r'(?i)contrat à durée indéterminée',                'CDD'),
    (r'(?i)cdd',                'CDD'),
    (r'(?i)contrat à durée déterminée',                'CDD'),
    (r'(?i)profession\s+lib',   'Profession libérale'),
    (r'(?i)intér?im',           'Intérim'),
    (r'(?i)saisonnier',           'Saisonnier'),
    (r'(?i)profession commerciale',     'Profession commerciale'),
    (r'(?i)franchise',     'Franchise')    
]

# Apply
df_normalize = normalize_contracts(df, patterns)

df_normalize_contract_counts_by_source = df_normalize.groupby('source')['contract_normalized'].value_counts()
print(f"Modalités de contrats FT = {len(df_normalize_contract_counts_by_source.loc['FT'])}")
print(f"Modalités de contrats WTTJ = {len(df_normalize_contract_counts_by_source.loc['WTTJ'])}")

display(df_normalize_contract_counts_by_source.head(30))

plot_by_source(df_normalize, 'FT', 'Répartition des types de contrat' ,  'contract_normalized')
plot_by_source(df_normalize, 'FT', 'Répartition des types de contrat' ,  'contract_normalized')



### Expérience

In [ ]:
experience_level_counts_by_source = df.groupby('source')['experience_level'].value_counts()
experience_description_counts_by_source = df.groupby('source')['experience_description'].value_counts()

### Avant normalisation

**FT**
- experienceExige => D : débutant accepté, E : l’expérience est exigée, S : l’expérience est souhaitée
- experienceLibelle => Libellé de l’expérience ex : Débutant accepté / 1 ans ... 
- experienceCommentaire => Commentaire sur l’expérience. Ex: Expérience dans la vente souhaitée

On a pas la bonne correspondance de colonne.
Il faut prendre `experience_level` pour `wttj` et `experience_description` pour FT


In [ ]:
display(experience_level_counts_by_source.head(30))


In [ ]:
display(experience_description_counts_by_source.head(30))

In [ ]:
# Définition des tranches d'expérience avec leurs indices
EXPERIENCE_LEVELS = [
    (0, 'Débutant',    [r'(?i)débutant', r'(?i)0 an', r'(?i)sans expérience']),
    (1, '0-1 an',      [r'(?i)^1 an', r'(?i)^6 mois', r'(?i)^1 mois', r'(?i)^3 mois', r'(?i)less_than_6_months', r'(?i)6_months_to_1_year']),
    (2, '1-2 ans',     [r'(?i)^2 an', r'(?i)1_to_2_years']),
    (3, '2-3 ans',     [r'(?i)^3 an', r'(?i)^24 mois', r'(?i)2_to_3_years' ]),
    (4, '3-5 ans',     [r'(?i)^4 an', r'(?i)^5 an', r'(?i)4_to_5_years', r'(?i)3_to_4_years']),
    (5, '5-10 ans',    [r'(?i)^6 an', r'(?i)^7 an', r'(?i)^8 an', r'(?i)^9 an', r'(?i)^10 an', r'(?i)5_to_7_years', r'(?i)7_to_10_years']),
    (6, '10+ ans',     [r'(?i)^1[1-9] an', r'(?i)^[2-9][0-9] an', r'(?i)10_to_15_years', r'(?i)more_than_15_years']),
    (-1, 'Non précisé', [r'(?i)expérience exigée', r'(?i)expérience souhaitée']),
   
]

def normalize_experience(df, experience_col):
    """
    Normalise les niveaux d'expérience :
    - experience_normalized : label lisible (ex: '0-1 an')
    - experience_index      : indice numérique (ex: 1) pour trier/comparer
    - experience_detail     : valeur originale nettoyée
    """

    def extract_experience(value):
        if pd.isna(value):
            #To DEBUG
            return -1, 'NAN', None
            #return -1, 'Non précisé', None
        
        str_value = str(value).strip()
        
        for index, label, patterns in EXPERIENCE_LEVELS:
            for pattern in patterns:
                if re.search(pattern, str_value):
                    # Détail = valeur originale
                    return index, label, str_value
        
        #return -1, 'Non précisé', str_value
        # To debug
        return -1, value, str_value
    

    results = df[experience_col].apply(extract_experience)
    
    df = df.copy()
    df['experience_index']      = results.apply(lambda x: x[0])
    df['experience_normalized'] = results.apply(lambda x: x[1])
    df['experience_detail']     = results.apply(lambda x: x[2])
    
    return df


In [ ]:
def get_experience_col(row):
    if row['source'] == 'FT':
        return row['experience_description']
    elif row['source'] == 'WTTJ':
        return row['experience_level']
    else:
        return None 

df_normalize['experience_source_composite'] = df.apply(get_experience_col, axis=1)

# Utilisation
df_normalize = normalize_experience(df_normalize,'experience_source_composite')

# Vérification
#df_normalize[['experience_source_composite', 'experience_index', 'experience_normalized', 'experience_detail']].head(20)

display(df_normalize.groupby('source')['experience_normalized'].value_counts())

## ROME

In [ ]:
rome_count = df['rome_code'].value_counts()


## Statistics from helper

In [ ]:
if df is not None and not df.empty:
    merge_utils.print_statistics(df)
else    :
    logger.warning("⚠️ Aucune donnée chargée pour les statistiques FT/WTTJ fusionnées")